<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L09-streaming-monitor-in-c/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L09-streaming-monitor-in-c/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/model-risk/lessons/P04-L09-streaming-monitor-in-c/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have, and a C or C++ compiler (`clang` or `gcc`). The cell
below fetches the files it needs beside it, and does nothing where they are already present.
On Kaggle, switch Internet on in the notebook's settings first; Kaggle allows that only for
phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = ["Makefile", "lesson.c"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/model-risk/lessons/P04-L09-streaming-monitor-in-c/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P04-L09 · Monitoring a portfolio that does not fit in memory

**You will build:** a monthly stability monitor in C that reads a score extract many times
larger than the memory it is allowed, in one pass, through a pipe — counting each score into
module 1's baseline bins, keeping a running PSI floored exactly as module 1 floors it,
drawing a reproducible random sample of the tail, and raising a breach that names the bin
and the checkpoint that tripped it. Then the harness measures the peak memory your monitor
actually used, from outside it, and holds it to a ceiling.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, no GPU, no download, no network
· **Prerequisites:** T00-L01 (the tier gate and how peak memory is measured), P04-L01 (the
validation suite, whose binning and PSI this lesson carries over unchanged)

By the end you will be able to:
1. Implement module 1's binning and floored PSI in C over bin counts, and reproduce an
   in-memory numpy reference: the counts exactly, the PSI within a tolerance you can state.
2. Implement a single-pass monitor that reads a pipe once through a fixed buffer, rejects
   every score that is not a probability — NaN included — and reports a truncated record.
3. Implement Algorithm R over the tail with a generator shared by C and Python, and
   reproduce the sample bit for bit.
4. Measure the monitor's peak resident memory from outside it, convert `ru_maxrss` for the
   platform that produced it, and hold the monitor under a ceiling set over its own baseline.
5. Explain why the harness measures a forked worker rather than its own high-water mark.

Five of the six exercises are in **`lesson.c`**. This notebook builds it, feeds it, measures
it and grades it. The sixth, in Python, is the unit conversion without which no memory
figure in this lesson means anything.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import atexit
import contextlib
import hashlib
import io
import math
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import time
import traceback
from pathlib import Path
from typing import Callable, NamedTuple

import numpy as np

print("numpy", np.__version__, "· python", sys.version.split()[0])
_LESSON_T0 = time.perf_counter()

# True in a notebook and when this file is run as a script; False when the autograder imports
# it. Every check below is called under this guard, so importing the lesson runs nothing.
_IS_MAIN = __name__ == "__main__"

try:
    LESSON_DIR = Path(__file__).resolve().parent
except NameError:  # a notebook has no __file__
    LESSON_DIR = Path.cwd()

C_SRC = "lesson.c"
BIN = "lesson_bin"

# The contract, mirrored from lesson.c. Section 1 checks the binary agrees before any number
# it prints is trusted.
STREAM_BUFFER_BYTES = 65536     # the monitor's whole allowance for data in flight
RECORD_BYTES = 16               # uint64 account id, then a float64 score, little-endian
CHECKPOINT_EVERY = 262144       # records read between two running-PSI checkpoints
MAX_BINS = 64
RESERVOIR_MAX = 4096

# Module 1's constants, carried over unchanged.
N_BINS = 10
PSI_FLOOR = 1e-6
PSI_THRESHOLD = 0.25

# The data. Synthetic, generated below, never downloaded.
SEED = 20260909
GAMMA = 0x9E3779B97F4A7C15
MASK64 = (1 << 64) - 1
N_DEV = 200_000                 # the development sample the baseline was cut on
N_MONTH = 1 << 22               # records in the month's extract
NEW_CHANNEL_FROM = 1_500_000    # the record at which a new origination channel opens
NEW_CHANNEL_SHARE = 0.6         # the share of later records that come through it
ACCOUNT_BASE = 100_000_000_000
TAIL_BIN = N_BINS - 2           # the tail: the top two baseline deciles
RESERVOIR_K = 1000
RESERVOIR_SEED = 4242424242

# The memory gate. The ceiling is the worker's own measured baseline plus this much.
MEMORY_HEADROOM_MIB = 4.0
# Two correct PSI implementations disagree only through log() and the order of one short sum;
# this is how many units in the last place, per bin, the comparison allows for that.
PSI_TOLERANCE_ULPS = 64

REC_DTYPE = np.dtype([("account_id", "<u8"), ("score", "<f8")])

DATA_NOTE = ("SYNTHETIC. Every account and every score in this lesson is generated in this "
             "notebook from a counter-based splitmix64 stream seeded with 20260909. No real "
             "portfolio, applicant or lending decision is represented anywhere.")

_BUILD = None
_BUILT_FROM = None
_CACHE: dict = {}
_WORK: list = []                # the temporary directory, once one exists


def work_dir() -> Path:
    """The lesson's scratch directory, under the system temp directory, created on first use.

    Everything this lesson generates at run time — the month's extract, the baseline profile —
    goes here and is deleted when the notebook finishes or the interpreter exits. Never into
    the lesson directory: the month's extract written there would be uploaded by any sync
    client watching it, while it was still being written.
    """
    if not _WORK:
        _WORK.append(Path(tempfile.mkdtemp(prefix="p04l09-")))
        atexit.register(clean_up)
    return _WORK[0]


def clean_up() -> None:
    """Delete the scratch directory and everything in it."""
    while _WORK:
        shutil.rmtree(_WORK.pop(), ignore_errors=True)


def build(verbose: bool = True):
    """Compile the C source with make. Cached, once per SAVED version of the source."""
    global _BUILD, _BUILT_FROM
    src = LESSON_DIR / C_SRC
    stamp = src.stat().st_mtime_ns if src.exists() else None
    if _BUILD is None or stamp != _BUILT_FROM:
        proc = subprocess.run(["make", "-C", str(LESSON_DIR), f"PYTHON={sys.executable}",
                               f"SRC={C_SRC}", f"BIN={BIN}"],
                              capture_output=True, text=True, timeout=300)
        _BUILD = (proc.returncode == 0, (proc.stdout + proc.stderr).strip())
        _BUILT_FROM = stamp
        _CACHE.clear()
    ok, out = _BUILD
    if verbose:
        print("build OK" if ok else "BUILD FAILED\n" + out)
    return ok, out


def _binary() -> str:
    ok, out = build(verbose=False)
    if not ok:
        raise RuntimeError("the C build failed; run build() to see the compiler output\n" + out)
    return str(LESSON_DIR / BIN)


def _num(v: str):
    try:
        return int(v)
    except ValueError:
        return float(v)


def parse_report(text: str) -> dict:
    """Parse what the binary prints: `@ key=value` lines, plus tagged rows."""
    rep = {"counts": {}, "contrib": {}, "checkpoint_rows": [], "samples": [], "bins": []}
    for line in text.splitlines():
        if line.startswith("@ "):
            key, value = line[2:].split("=", 1)
            rep[key] = _num(value)
            continue
        parts = line.split()
        tag = parts[0] if parts else ""
        if tag == "count":
            rep["counts"][int(parts[1])] = int(parts[2])
        elif tag == "contrib":
            rep["contrib"][int(parts[1])] = float(parts[2])
        elif tag == "checkpoint":
            rep["checkpoint_rows"].append((int(parts[1]), int(parts[2]), float(parts[3]),
                                           int(parts[4])))
        elif tag == "sample":
            rep["samples"].append((int(parts[1]), int(parts[2]), float(parts[3])))
        elif tag == "bin":
            rep["bins"].append((parts[1], int(parts[2])))
    return rep


def _raise_for(code: int, command: str, err: str) -> None:
    if code == 2:
        raise NotImplementedError(err.strip() or f"{command}: a C exercise is still a stub")
    if code != 0:
        raise RuntimeError(f"{BIN} {command} exited {code}\n{err.strip()}")


def run_c(command: str, *args) -> dict:
    """Run one short command of the binary and parse what it printed."""
    proc = subprocess.run([_binary(), command, *map(str, args)], cwd=LESSON_DIR,
                          capture_output=True, text=True, timeout=300)
    _raise_for(proc.returncode, command, proc.stderr)
    return parse_report(proc.stdout)


def _write_all(pipe, data) -> None:
    view = memoryview(data)
    while view:
        view = view[pipe.write(view):]


def _feed(pipe, feed, limit, write_size, pause) -> None:
    """Write `feed` (a path, or bytes) into the worker's stdin pipe, then close the pipe.

    With `pause`, wait that many seconds after each write of a bytes feed, so the reader has
    drained the pipe before the next piece arrives: a read then ends exactly where a write
    ended, which is how a real producer delivers a stream a little at a time.
    """
    try:
        if isinstance(feed, (bytes, bytearray)):
            data = memoryview(feed)[: len(feed) if limit is None else limit]
            for off in range(0, len(data), write_size):
                _write_all(pipe, data[off: off + write_size])
                if pause:
                    time.sleep(pause)
        else:
            left = limit
            with open(feed, "rb") as f:
                while left is None or left > 0:
                    chunk = f.read(write_size if left is None else min(write_size, left))
                    if not chunk:
                        break
                    _write_all(pipe, chunk)
                    if left is not None:
                        left -= len(chunk)
    except BrokenPipeError:
        pass  # the worker stopped reading: an unfinished stub exits before the stream ends
    finally:
        with contextlib.suppress(BrokenPipeError):
            pipe.close()


def run_worker(command: str, *args, feed=None, limit=None, write_size: int = 1 << 20,
               pause: float = 0.0) -> dict:
    """Run a measured command. With `feed`, stream it into the binary through a PIPE on stdin.

    The pipe is the point: it can be read exactly once and cannot seek, so a monitor that
    works at all works in a single pass. The binary forks a worker, and reports the worker's
    peak resident memory as `worker_maxrss_raw`, in whatever unit this platform uses.
    """
    with tempfile.TemporaryFile() as out, tempfile.TemporaryFile() as err:
        proc = subprocess.Popen([_binary(), command, *map(str, args)], cwd=LESSON_DIR,
                                stdin=subprocess.PIPE if feed is not None else subprocess.DEVNULL,
                                stdout=out, stderr=err, bufsize=0)
        if feed is not None:
            _feed(proc.stdin, feed, limit, write_size, pause)
        try:
            code = proc.wait(timeout=300)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
            raise RuntimeError(f"{BIN} {command} did not finish in 300 s") from None
        out.seek(0)
        err.seek(0)
        text, message = out.read().decode(), err.read().decode()
    _raise_for(code, command, message)
    return parse_report(text)


def make_test():
    """Run the C self-test — what `make test` runs. Returns (exit code, combined output)."""
    proc = subprocess.run([_binary(), "selftest"], cwd=LESSON_DIR, capture_output=True,
                          text=True, timeout=300)
    return proc.returncode, (proc.stdout + proc.stderr).rstrip()


# The six exercises, and the function each asks for. `_try` records the latest outcome of
# every check in `_STATUS`; the progress board at the foot of the notebook reads it.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("maxrss_mib",),
    "exercise 2": ("bin_index",),
    "exercise 3": ("psi_from_counts",),
    "exercise 4": ("breach_bin",),
    "exercise 5": ("reservoir_offer",),
    "exercise 6": ("monitor_stream",),
}
_IN_C = {"bin_index", "psi_from_counts", "breach_bin", "reservoir_offer", "monitor_stream"}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_FAILED_CHECKS: list[str] = []


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (psi_from_counts)"; several -> "exercises 3, 4 and 6"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _stub_of(exc: NotImplementedError) -> str:
    """The function whose stub raised: a C stub names itself in the binary's message, and a
    Python stub is the innermost frame of the traceback."""
    m = re.search(r"(\w+)\(\) is still a stub", str(exc))
    if m:
        return m.group(1)
    return traceback.extract_tb(exc.__traceback__)[-1].name


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = _stub_of(exc)
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        where = "in lesson.c" if stub in _IN_C else "above"
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() {where}, then re-run "
                  "this cell.")
        elif any(stub in funcs for funcs in _EXERCISES.values()):
            print(f"{label}: skipped — {stub}() is not implemented yet.")
        else:
            print(f"{label}: not started — {exc}")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


if _IS_MAIN:
    build()

## 1. The monthly run does not get a bigger machine

Once a month, somebody scores the whole book and asks whether the population still looks
like the one the model was built on. The 2026 interagency guidance, carried by the OCC as
Bulletin 2026-13, names "model validation and monitoring" among the practices it covers
(`claims.yaml`). It says nothing about how much memory the monitoring job gets, and in
practice the answer is: the same box as last month, while the book has grown.

So this lesson's monitor has a **fixed allowance for data in flight**, declared in
`lesson.c` and printed below, and an extract many times that size. It reads the extract
once, from a pipe, and must still reproduce module 1's in-memory PSI: the bin counts
exactly, the index itself within a tolerance the notebook states. And the harness does not
take its word for the memory: it measures it.

First, ask the binary for the contract, and check it agrees with this notebook. Nothing
below trusts a number from a binary built to a different contract.

In [ ]:
def _show_contract() -> None:
    facts = run_c("facts")
    mirrored = {"stream_buffer_bytes": STREAM_BUFFER_BYTES, "record_bytes": RECORD_BYTES,
                "checkpoint_every": CHECKPOINT_EVERY, "max_bins": MAX_BINS,
                "reservoir_max": RESERVOIR_MAX, "psi_floor": PSI_FLOOR,
                "psi_threshold": PSI_THRESHOLD}
    wrong = {k: (facts[k], v) for k, v in mirrored.items() if facts[k] != v}
    assert not wrong, (f"lesson.c and this notebook disagree on {wrong} (C, notebook). Put the "
                       "constants back as they were: every check below assumes them.")
    assert facts["little_endian"] == 1 and facts["sizeof_record"] == RECORD_BYTES, (
        "the extract is little-endian 16-byte records, and this machine or build disagrees")
    print(f"  allowance for data in flight  {facts['stream_buffer_bytes']} bytes "
          f"({facts['stream_buffer_bytes'] // 1024} KiB)")
    print(f"  one record                    {facts['record_bytes']} bytes: account id, score")
    print(f"  running PSI every             {facts['checkpoint_every']} records")
    print(f"  PSI floor, threshold          {facts['psi_floor']!r}, {facts['psi_threshold']!r}"
          "  (module 1's)")
    print(f"  the Monitor struct            {facts['monitor_struct_bytes']} bytes, whatever "
          "the length of the month")
    print(f"\n  {DATA_NOTE}")


if _IS_MAIN:
    _try("the contract", _show_contract)

## 2. The baseline, carried over from module 1

Module 1 cut the PSI edges **once**, at equally spaced quantiles of the development sample,
and scored every later sample against them — because edges re-cut on each month's own data
make any drift read as stable. The functions below are module 1's, carried over with the
same names and the same arithmetic, so that "module 1's PSI" means one thing in this
programme.

A monitor does not hold the development sample. It holds a **profile**: the edges, and how
many development records fell in each bin. That is all PSI ever needed from the baseline,
and it fits in a few hundred bytes. The cell writes it to a file in the system's temporary
directory; the C monitor loads it from there, as a production job would load it from
wherever the model's documentation lives.

In [ ]:
def quantile_edges(x: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Module 1's edges: equally spaced quantiles of `x`, outer edges opened to infinity."""
    edges = np.quantile(np.asarray(x, dtype=float), np.linspace(0.0, 1.0, n_bins + 1))
    edges = np.unique(edges)
    edges[0] = -np.inf
    edges[-1] = np.inf
    return edges


class StabilityResult(NamedTuple):
    """Module 1's PSI result: the total, and the per-bin arithmetic that produced it."""

    psi: float
    contributions: np.ndarray
    expected_pct: np.ndarray
    actual_pct: np.ndarray


def population_stability_index(expected: np.ndarray, actual: np.ndarray,
                               edges: np.ndarray, floor: float = PSI_FLOOR) -> StabilityResult:
    """Module 1's PSI of `actual` against baseline `expected` on fixed `edges`. GIVEN.

    Bin i is [edges[i], edges[i+1]); values outside the edges fall into the end bins. Shares,
    not counts, floored at `floor` on both sides before the logarithm.
    """
    edges = np.asarray(edges, dtype=float)
    if edges.size < 2:
        raise ValueError(f"edges needs at least two values, got {edges.size}")
    if not np.all(np.diff(edges) > 0):
        raise ValueError("edges must be strictly increasing")
    n_bins = edges.size - 1

    def _shares(x: np.ndarray) -> np.ndarray:
        v = np.asarray(x, dtype=float)
        if v.size == 0:
            raise ValueError("population_stability_index needs a non-empty sample on both sides")
        idx = np.clip(np.searchsorted(edges, v, side="right") - 1, 0, n_bins - 1)
        return np.bincount(idx, minlength=n_bins).astype(float) / v.size

    e_pct = _shares(expected)
    a_pct = _shares(actual)
    e_safe = np.maximum(e_pct, floor)
    a_safe = np.maximum(a_pct, floor)
    contributions = (a_safe - e_safe) * np.log(a_safe / e_safe)
    return StabilityResult(psi=float(contributions.sum()), contributions=contributions,
                           expected_pct=e_pct, actual_pct=a_pct)


def module1_bins(x: np.ndarray, edges: np.ndarray) -> np.ndarray:
    """Module 1's binning rule on its own: which bin each value of `x` falls in."""
    n_bins = len(edges) - 1
    return np.clip(np.searchsorted(edges, np.asarray(x, dtype=float), side="right") - 1,
                   0, n_bins - 1)


def psi_from_counts_reference(expected_counts, actual_counts,
                              floor: float = PSI_FLOOR) -> StabilityResult:
    """Module 1's arithmetic, applied to bin COUNTS rather than to samples. GIVEN.

    Section 3 checks, to the bit, that this and population_stability_index agree on the
    month — so what your C function is compared against really is module 1's PSI.
    """
    e = np.asarray(expected_counts, dtype=np.int64)
    a = np.asarray(actual_counts, dtype=np.int64)
    if e.sum() <= 0 or a.sum() <= 0:
        nan = np.full(e.size, np.nan)
        return StabilityResult(float("nan"), nan, nan, nan)
    e_pct = e.astype(float) / int(e.sum())
    a_pct = a.astype(float) / int(a.sum())
    e_safe = np.maximum(e_pct, floor)
    a_safe = np.maximum(a_pct, floor)
    contributions = (a_safe - e_safe) * np.log(a_safe / e_safe)
    return StabilityResult(float(contributions.sum()), contributions, e_pct, a_pct)


def breach_reference(psi: float, contributions) -> int:
    """The breach rule, in Python: strictly over the threshold, largest bin, lowest on ties."""
    if not psi > PSI_THRESHOLD:
        return -1
    return int(np.argmax(np.asarray(contributions)))   # argmax returns the FIRST maximum


def psi_tolerance(contributions) -> float:
    """How far apart two CORRECT PSI implementations may land, as a stated formula.

    Both sides compute the same shares from the same integer counts, bit for bit. They differ
    in two places only: C's log() and numpy's are different implementations, each good to
    about one unit in the last place, and the two sums add the bins in different orders. So
    the allowance has the number of bins and the size of the figures in it, as module 6 said
    a tolerance must, with PSI_TOLERANCE_ULPS per bin of room for both effects.
    """
    c = np.nan_to_num(np.abs(np.asarray(contributions, dtype=float)))
    return PSI_TOLERANCE_ULPS * c.size * 2.0 ** -53 * max(1.0, float(c.sum()))


def _mix(z: np.ndarray) -> np.ndarray:
    z = (z ^ (z >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
    z = (z ^ (z >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
    return z ^ (z >> np.uint64(31))


def _draws(seed: int, start: int, count: int) -> np.ndarray:
    """Counter-based splitmix64: draw k is mix(seed + (k + 1) * GAMMA)."""
    k = np.arange(start, start + count, dtype=np.uint64)
    return _mix(np.uint64(seed) + (k + np.uint64(1)) * np.uint64(GAMMA))


def _u53(z: np.ndarray) -> np.ndarray:
    """A uniform in [0, 1) from the top 53 bits: exact, the same double on every machine."""
    return (z >> np.uint64(11)).astype(np.float64) * 2.0 ** -53


def development_sample(n: int = N_DEV, seed: int = SEED + 1) -> np.ndarray:
    """The development sample's scores. SYNTHETIC: a product of two uniforms, skewed low,
    the rough shape of a probability-of-default score. Only exact IEEE operations are used,
    so every machine and every numpy generates the same bits."""
    z = _draws(seed, 0, 2 * n)
    return _u53(z[0::2]) * _u53(z[1::2])


def write_profile(path: Path, edges, counts, tail_bin: int) -> Path:
    """Write a baseline profile in the text format lesson.c's load_profile() reads."""
    lines = ["p04l09-profile 1", f"n_bins {len(counts)}", f"tail_bin {tail_bin}",
             "edges " + " ".join(repr(float(e)) for e in edges),
             "counts " + " ".join(str(int(c)) for c in counts)]
    Path(path).write_text("\n".join(lines) + "\n")
    return Path(path)


def profile_file() -> Path:
    """The baseline profile's path, rewritten first if the scratch directory was cleaned."""
    path = work_dir() / "profile.txt"
    if not path.exists():
        write_profile(path, EDGES, DEV_COUNTS, TAIL_BIN)
    return path


DEV = development_sample()
EDGES = quantile_edges(DEV, N_BINS)
DEV_COUNTS = np.bincount(module1_bins(DEV, EDGES), minlength=N_BINS)

if _IS_MAIN:
    print(f"development sample: {N_DEV} records, cut once into {N_BINS} bins")
    print(f"  {'bin':>3}  {'from':>9}  {'to':>9}  {'records':>8}")
    for i in range(N_BINS):
        print(f"  {i:>3}  {EDGES[i]:>9.6f}  {EDGES[i + 1]:>9.6f}  {DEV_COUNTS[i]:>8d}"
              + ("   <- tail" if i >= TAIL_BIN else ""))
    print(f"\nthe profile file is {profile_file().stat().st_size} bytes, in the temporary "
          "directory")

## 3. The month's extract, and the reference that is allowed to hold it

The month is generated from a counter-based splitmix64 stream, out of operations that are
exact in IEEE arithmetic, so it is the same bits on every machine and in every numpy. Two
things are planted in it, and the notebook does not tell you how big either one is:

- part-way through the month a **new origination channel** opens, and a share of the
  accounts after that point come through it with riskier scores;
- a **feed defect**: a scattering of records whose score is not a probability at all — NaN,
  negative, a hair above one — beside records sitting exactly on 0 and on 1, which are fine.

The file is written to the temporary directory as raw 16-byte records. Then the **in-memory
reference** reads the whole month into numpy and runs module 1's PSI on it. The reference is
allowed the memory; it is the thing you check against, not the thing you ship.

In [ ]:
def month_extract(n: int = N_MONTH, seed: int = SEED, chunk: int = 1 << 20) -> np.ndarray:
    """The month's records, as a structured array of (account_id, score). SYNTHETIC."""
    out = np.empty(n, dtype=REC_DTYPE)
    q32 = np.uint64(round(NEW_CHANNEL_SHARE * 2 ** 32))
    for start in range(0, n, chunk):
        count = min(chunk, n - start)
        z = _draws(seed, 3 * start, 3 * count)
        ua, ub, c = _u53(z[0::3]), _u53(z[1::3]), z[2::3]
        base = ua * ub
        pos = np.arange(start, start + count)
        new_channel = (pos >= NEW_CHANNEL_FROM) & ((c >> np.uint64(32)) < q32)
        score = np.where(new_channel, np.sqrt(np.sqrt(base)), base)
        defect = c & np.uint64(0xFFFF)
        score = np.where(defect == 0, np.nan, score)
        score = np.where(defect == 1, -(score + 0.5), score)
        score = np.where(defect == 2, 1.0 + 2.0 ** -52, score)
        score = np.where(defect == 3, 1.0, score)
        score = np.where(defect == 4, 0.0, score)
        score = np.where(defect == 5, -0.0, score)
        out["account_id"][start:start + count] = np.uint64(ACCOUNT_BASE) + pos.astype(np.uint64)
        out["score"][start:start + count] = score
    return out


def splitmix64_next(state: int) -> tuple[int, int]:
    """One step of splitmix64 in plain Python: returns (new state, output). The same four
    lines as splitmix64_next() in lesson.c, on Python's exact integers."""
    state = (state + GAMMA) & MASK64
    z = state
    z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & MASK64
    z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & MASK64
    return state, z ^ (z >> 31)


def splitmix64_outputs(seed: int, count: int) -> np.ndarray:
    """The first `count` outputs of splitmix64 seeded with `seed`, vectorised: output i
    (counting from 1) is mix(seed + i * GAMMA), because the state only ever adds GAMMA."""
    i = np.arange(1, count + 1, dtype=np.uint64)
    return _mix(np.uint64(seed) + i * np.uint64(GAMMA))


def reservoir_reference(ids, scores, k: int, seed: int):
    """Algorithm R over (ids, scores) in order, drawing from splitmix64 exactly as the C does.

    Returns (kept ids, kept scores, records offered, final generator state). Vectorised: the
    draw for offer t >= k depends only on t, so all the draws are made at once, and each slot
    ends up holding the LAST offer that landed in it.
    """
    ids = np.asarray(ids, dtype=np.uint64)
    scores = np.asarray(scores, dtype=np.float64)
    t = ids.size
    keep = min(t, k)
    out_ids, out_scores = ids[:keep].copy(), scores[:keep].copy()
    state = seed
    if t > k:
        pos = np.arange(k, t, dtype=np.uint64)
        j = splitmix64_outputs(seed, t - k) % (pos + np.uint64(1))
        hit = np.nonzero(j < np.uint64(k))[0]
        last = np.full(k, -1, dtype=np.int64)
        np.maximum.at(last, j[hit].astype(np.int64), hit + k)
        took = last >= 0
        out_ids[took] = ids[last[took]]
        out_scores[took] = scores[last[took]]
        state = (seed + (t - k) * GAMMA) & MASK64
    return out_ids, out_scores, t, state


def month_file() -> Path:
    """The month's extract on disk, rewritten first if the scratch directory was cleaned."""
    path = work_dir() / "month.bin"
    if not path.exists():
        MONTH.tofile(path)
    return path


_t0 = time.perf_counter()
MONTH = month_extract()
month_file()
_GEN_SECONDS = time.perf_counter() - _t0

if _IS_MAIN:
    size = month_file().stat().st_size
    print(f"the month: {N_MONTH} records, {size} bytes on disk, written in "
          f"{_GEN_SECONDS:.2f} s")
    print(f"that is {size // STREAM_BUFFER_BYTES} times the monitor's allowance for data in "
          "flight")

In [ ]:
def in_memory_reference(month: np.ndarray = MONTH, k: int = RESERVOIR_K,
                        seed: int = RESERVOIR_SEED) -> dict:
    """Everything the monitor must reproduce, computed with the whole month in memory."""
    scores = month["score"]
    ids = month["account_id"]
    n = scores.size
    valid = (scores >= 0.0) & (scores <= 1.0)
    idx = module1_bins(np.where(valid, scores, 0.0), EDGES)
    counts = np.bincount(idx[valid], minlength=N_BINS)
    whole = population_stability_index(DEV, scores[valid], EDGES)   # module 1, verbatim
    # The running PSI: valid counts per bin at every CHECKPOINT_EVERY records read. One pass
    # of bincount over (block, bin) pairs, then a cumulative sum down the blocks.
    block = np.arange(n) // CHECKPOINT_EVERY
    n_blocks = -(-n // CHECKPOINT_EVERY)
    table = np.bincount(block[valid] * N_BINS + idx[valid],
                        minlength=n_blocks * N_BINS).reshape(n_blocks, N_BINS).cumsum(axis=0)
    checkpoints, first = [], (-1, -1)
    for b in range(n // CHECKPOINT_EVERY):
        r = psi_from_counts_reference(DEV_COUNTS, table[b])
        bin_ = breach_reference(r.psi, r.contributions)
        checkpoints.append(((b + 1) * CHECKPOINT_EVERY, int(table[b].sum()), r.psi, bin_))
        if bin_ >= 0 and first[0] < 0:
            first = ((b + 1) * CHECKPOINT_EVERY, bin_)
    final_bin = breach_reference(whole.psi, whole.contributions)
    if final_bin >= 0 and first[0] < 0:
        first = (n, final_bin)
    tail = valid & (idx >= TAIL_BIN)
    kept_ids, kept_scores, seen, state = reservoir_reference(ids[tail], scores[tail], k, seed)
    return {"records_read": n, "valid": int(valid.sum()), "rejected": int((~valid).sum()),
            "nan": int(np.isnan(scores).sum()), "counts": counts, "psi": whole.psi,
            "contributions": whole.contributions, "expected_pct": whole.expected_pct,
            "actual_pct": whole.actual_pct, "breach_bin": final_bin, "checkpoints": checkpoints,
            "first_breach": first, "tail_seen": seen, "sample_ids": kept_ids,
            "sample_scores": kept_scores, "held_bytes": month.nbytes}


REF = in_memory_reference()


def _show_reference() -> None:
    same = psi_from_counts_reference(DEV_COUNTS, REF["counts"])
    assert same.psi == REF["psi"] and np.array_equal(same.contributions, REF["contributions"]), (
        "module 1's PSI on the samples and on the counts should be the same doubles")
    print(f"  records {REF['records_read']}: {REF['valid']} valid, {REF['rejected']} rejected "
          f"({REF['nan']} of them NaN)")
    print(f"  module 1's PSI on the whole month: {REF['psi']:.6f}   (threshold "
          f"{PSI_THRESHOLD})")
    print(f"  the same PSI from the bin counts alone: identical to the last bit")
    b = REF["breach_bin"]
    print(f"  breach: bin {b}, share {REF['expected_pct'][b]:.4f} in development against "
          f"{REF['actual_pct'][b]:.4f} this month")
    print(f"  first breached at the checkpoint after record {REF['first_breach'][0]}")
    print(f"  tail records (bins {TAIL_BIN} and up): {REF['tail_seen']}, "
          f"{REF['tail_seen'] / REF['valid']:.1%} of the valid records; sample of "
          f"{RESERVOIR_K} drawn")
    print(f"  the reference held the whole month as arrays: {REF['held_bytes']} bytes of "
          "records, before any working copies")


if _IS_MAIN:
    _try("the in-memory reference", _show_reference)

## 4. Exercise 1 — `maxrss_mib()`, in Python

Every memory figure in this lesson comes from `getrusage()`'s `ru_maxrss`: the peak resident
set of a process, the operating system's own high-water mark (T00-L01). Its **unit is the
operating system's choice**. The Linux manual page says KiB; the page shipped with current
macOS says bytes, while Apple's older archived copy of the same page still says kilobytes
(`claims.yaml` quotes all three). A factor of 1024 either way, and a memory ceiling in the
wrong unit passes every monitor or fails every one.

The course's CI runs on Linux; you may well be on a Mac. So the conversion takes the name
of the system that took the reading, and refuses one it was not told about. The cell after
the check then measures the unit on your machine rather than believing any of the pages.

<details><summary>💡 Hint 1 — what to think about</summary>

There are exactly two systems this function knows, and they differ by one factor of 1024 in
the unit. A mebibyte is 1024 KiB, and a KiB is 1024 bytes — so one of the two needs one
division and the other needs two. Think about what a caller should get for a system the
function has never heard of: a guess is the dangerous answer. And the result has to be a
plain Python float, not an int and not a numpy scalar.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Lower-case the system name. Refuse a negative reading with ValueError. For "darwin" the
reading is in bytes, so divide by the number of bytes in a MiB; for "linux" it is in KiB, so
divide by the number of KiB in a MiB; for anything else raise ValueError naming the system.
Wrap the result in float() on the way out.

</details>

In [ ]:
def maxrss_mib(raw: int, system: str) -> float:
    """Convert a raw ru_maxrss reading into MiB, for the operating system that produced it.

    `system` is what platform.system() returns on the machine that took the reading —
    "Darwin" on macOS, "Linux" on Linux — compared without regard to case. The two disagree
    on the unit by a factor of 1024:
      * Linux reports KiB;
      * macOS reports bytes.
    Anything else raises ValueError: this function refuses to guess a unit it was not told,
    because a memory ceiling in the wrong unit passes everything or fails everything. A
    negative reading raises ValueError too.

    Worked example: 1.5 MiB of peak memory reads 1536 on Linux (KiB) and 1572864 on a Mac
    (bytes), so maxrss_mib(1536, "Linux") and maxrss_mib(1572864, "Darwin") are both 1.5.

    Returns a plain Python float, in MiB (2**20 bytes).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement maxrss_mib")


def _check_maxrss_mib() -> None:
    f = maxrss_mib
    assert f(1048576, "Darwin") == 1.0, (
        f"a Mac reading of 1048576 is 1048576 bytes, one MiB; you returned "
        f"{f(1048576, 'Darwin')!r} — macOS reports bytes, so divide by the bytes in a MiB")
    assert f(1024, "Linux") == 1.0, (
        f"a Linux reading of 1024 is 1024 KiB = 1.0 MiB; you returned {f(1024, 'Linux')!r} — "
        "Linux reports KiB, so divide once by 1024, not twice")
    assert f(1536, "Linux") == 1.5 and f(1572864, "Darwin") == 1.5, (
        f"1536 KiB and 1572864 bytes are both 1.5 MiB; you returned {f(1536, 'Linux')!r} and "
        f"{f(1572864, 'Darwin')!r} — use true division, so a fraction of a MiB survives")
    assert f(1536, "linux") == 1.5 and f(1572864, "DARWIN") == 1.5, (
        "the system name must be compared without regard to case: platform.system() says "
        "'Linux' and 'Darwin', sys.platform says 'linux' and 'darwin'")
    assert type(f(1024, "Linux")) is float, (
        f"return a plain Python float, not {type(f(1024, 'Linux')).__name__}")
    for bad in ("Windows", ""):
        try:
            f(1024, bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"maxrss_mib(1024, {bad!r}) must raise ValueError: no known "
                                 "unit, and a guessed unit is a wrong ceiling")
    try:
        f(-1, "Linux")
    except ValueError:
        pass
    else:
        raise AssertionError("a negative reading must raise ValueError")
    print("exercise 1 looks right")


if _IS_MAIN:
    _try("exercise 1", _check_maxrss_mib)

In [ ]:
def _show_unit_calibration() -> None:
    """Touch a known number of MiB in a worker and see what ru_maxrss did about it."""
    system = platform.system()
    idle = run_worker("touch", "--mib", 0)["worker_maxrss_raw"]
    busy = run_worker("touch", "--mib", 32)["worker_maxrss_raw"]
    seen = maxrss_mib(busy - idle, system)
    print(f"  this machine is {system!r}; a worker that touched 32 MiB raised its raw "
          f"ru_maxrss by ≈{busy - idle}, in whatever unit {system} uses")
    print(f"  your maxrss_mib reads that as {seen:.2f} MiB")
    assert 24.0 <= seen <= 48.0, (
        f"32 MiB touched came out as {seen:.2f} MiB, so the unit for {system!r} is wrong by "
        "about a factor of 1024 — check which system reports bytes and which reports KiB")
    print("  the calibration agrees with the unit the function assumes for this system")


if _IS_MAIN:
    _try("unit calibration", _show_unit_calibration, needs=("exercise 1",))

## 5. What holding the month costs

Before you write the monitor, see what the gate exists to catch. The binary has a `hold`
command: a "monitor" that reads the whole pipe into a buffer that doubles as it fills, and
only then looks at it. It is the obvious way to write the job, and it works — on the month it
was tested on.

Every measured command runs in a **worker** the binary forks, and the binary reports the
worker's peak as `wait4()` hands it back. The `baseline` command is the same worker doing
nothing but setting up: the process's own footprint before the first byte of the month
arrives. The ceiling is that baseline plus `MEMORY_HEADROOM_MIB` — honest headroom, many
times what a streaming monitor needs, and a small fraction of what holding the month costs.

In [ ]:
def memory_ceiling() -> tuple[float, float]:
    """(the worker's measured baseline in MiB, the ceiling in MiB)."""
    raw = run_worker("baseline", "--profile", profile_file(), "--k", RESERVOIR_K,
                     "--seed", RESERVOIR_SEED)["worker_maxrss_raw"]
    base = maxrss_mib(raw, platform.system())
    return base, base + MEMORY_HEADROOM_MIB


def _show_what_holding_costs() -> None:
    base, ceiling = memory_ceiling()
    held = run_worker("hold", feed=month_file())
    peak = maxrss_mib(held["worker_maxrss_raw"], platform.system())
    print(f"  the worker's own baseline       {base:8.2f} MiB")
    print(f"  the ceiling (+ {MEMORY_HEADROOM_MIB:.0f} MiB headroom)   {ceiling:8.2f} MiB")
    print(f"  a worker that held the month    {peak:8.2f} MiB   ({held['held_bytes']} bytes "
          "held)")
    print(f"  over the ceiling by ≈{peak / ceiling:.0f}x. A monitor written that way fails the "
          "gate however right its PSI is.")


if _IS_MAIN:
    _try("what holding the month costs", _show_what_holding_costs, needs=("exercise 1",))

## 6. Exercise 2 — `bin_index()` in `lesson.c`

From here the exercises are in C. Open `lesson.c`; each stub's comment carries the
requirements and a worked example. Your feedback loop is `make test`, which the next cell
runs: it reports each C exercise on its own line, TODO until you fill it in.

The first is the one every later number rests on: which baseline bin a score lands in.
Module 1 answered it with `searchsorted(..., side="right")`, clipped. A monitor that bins a
score sitting exactly on an edge into the other bin is monitoring a different model.

<details><summary>💡 Hint 1 — what to think about</summary>

Module 1's rule is "count the edges at or below the score, minus one, then clip". Three
places go wrong: a score exactly ON an interior edge (it belongs to the bin whose lower edge
it is), a score below the first edge, and a score at or above the last one — which is how the
last bin comes to be closed on the right when the outer edges are finite. The outer edges in
the lesson's own profile are infinite, so the check also tests a profile whose are not.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Search the sorted edges for how many are less than or equal to the score — a binary search
keeps it logarithmic, a plain scan over at most a few dozen edges is also fine. Subtract one.
If that is below zero use zero; if it is past the last bin use the last bin. Compare with
"less than or equal", not "less than": that single character decides every on-edge score.

</details>

In [ ]:
def _show_selftest() -> None:
    code, out = make_test()
    print(out)
    print(f"\nexit code {code}  (0 = all pass, 2 = something is still a stub, 1 = a failure)")


def _bins_of(edges, values) -> list[int]:
    path = write_profile(work_dir() / "probe_profile.txt", edges, [1] * (len(edges) - 1),
                         len(edges) - 1)
    rows = run_c("bins", "--profile", path, *[repr(float(v)) for v in values])["bins"]
    return [b for _v, b in rows]


def _check_bin_index() -> None:
    cases = [
        ([-np.inf, 0.25, 0.5, 0.75, np.inf],
         [0.0, -0.0, 0.1, 0.25, math.nextafter(0.25, 0.0), math.nextafter(0.25, 1.0), 0.5,
          0.74, 0.75, 1.0]),
        ([0.1, 0.5, 0.9], [0.0, 0.1, 0.3, 0.5, 0.9, 1.0]),
        (list(EDGES), [0.0, 1.0] + [v for e in EDGES[1:-1] for v in
                                    (float(e), math.nextafter(float(e), 0.0),
                                     math.nextafter(float(e), 1.0))]),
    ]
    for edges, values in cases:
        got = _bins_of(edges, values)
        want = [int(b) for b in module1_bins(values, np.asarray(edges, dtype=float))]
        bad = [(v, g, w) for v, g, w in zip(values, got, want) if g != w]
        if not bad:
            continue
        v, g, w = bad[0]
        on_edge = float(v) in [float(e) for e in edges]
        why = ("a score exactly ON an interior edge belongs to the bin ABOVE it — module 1 "
               "bins with side='right', so compare with <=, not <" if on_edge and 0 < w
               else "a score below the first edge is clipped into bin 0, and one at or above "
               "the last edge into the last bin" if w in (0, len(edges) - 2)
               else "count the edges that are <= the score, then subtract one")
        raise AssertionError(f"on edges {[float(e) for e in edges]}, bin_index({float(v)!r}) "
                             f"returned {g} where module 1's rule gives {w} — {why}. "
                             f"({len(bad)} of {len(values)} probes disagree.)")
    print("exercise 2 looks right: every probe lands where module 1 puts it")


if _IS_MAIN:
    _try("make test", _show_selftest)
    _try("exercise 2", _check_bin_index)

## 7. Exercise 3 — `psi_from_counts()` in `lesson.c`

The monitor never holds a sample, only a count per bin. So module 1's PSI has to be computed
from two vectors of counts: each divided by its own total, both sides floored at `PSI_FLOOR`
before the logarithm, a contribution per bin, and the total. Same arithmetic, different
input — and section 3 already showed, to the bit, that the counts form IS module 1's PSI.

Your answer is compared with that reference within `psi_tolerance()`, which is stated in
the setup cell and has the number of bins and the size of the figures in it. It is not
compared bit for bit, and that is not a concession: C's `log` and numpy's `log` are two
different implementations, and module 6 of this programme is a whole lesson on why two
correct programs may disagree in the last place — and why the allowance must be stated.

<details><summary>💡 Hint 1 — what to think about</summary>

Three mistakes carry the marks. Dividing one integer count by an integer total in C is
integer division, and every share becomes zero. Flooring only one side, or flooring after
the logarithm, lets a bin that emptied out — or one that appeared from nothing — turn the
whole index into an infinity or a NaN. And an empty side has no shares at all, which is its
own answer rather than a division by zero.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

If either total is zero, fill every contribution with NaN and return NaN. Otherwise, for
each bin, turn both counts into doubles before dividing each by its own side's total, raise
each share to at least the floor, and form the difference of the two shares times the
natural logarithm of their ratio. Store it in the contributions array and add it to a
running total, bin by bin, then return the total.

</details>

In [ ]:
def _psi_c(expected, actual, floor: float = PSI_FLOOR) -> tuple[float, np.ndarray]:
    rep = run_c("psi", "--expected", ",".join(map(str, expected)),
                "--actual", ",".join(map(str, actual)), "--floor", repr(floor))
    return rep["psi"], np.array([rep["contrib"][i] for i in range(len(expected))])


def _check_psi_from_counts() -> None:
    cases = [([10, 20, 30], [10, 20, 30]), ([10, 10], [20, 20]), ([30, 50, 20], [20, 50, 30]),
             ([50, 50], [100, 0]), ([100, 0], [50, 50]), ([50, 50, 0], [100, 0, 0]),
             (list(DEV_COUNTS), list(REF["counts"])), (list(DEV_COUNTS), [7] * N_BINS)]
    for expected, actual in cases:
        got, contrib = _psi_c(expected, actual)
        want = psi_from_counts_reference(expected, actual)
        tol = psi_tolerance(want.contributions)
        assert math.isfinite(got), (
            f"expected {expected} against actual {actual} gave psi {got!r}; floor BOTH shares "
            "at the floor before the logarithm, or a bin that emptied out — or appeared from "
            "nothing — becomes an infinity, and a bin empty on both sides a NaN")
        assert abs(got - want.psi) <= tol, (
            f"expected {expected} against actual {actual}: you returned psi {got!r}, module 1's "
            f"arithmetic gives {want.psi!r} (tolerance {tol:.1e}). Check that each share is a "
            "double divided by its OWN side's total — two long longs divide as integers — and "
            "that the log is the natural one")
        worst = int(np.argmax(np.abs(contrib - want.contributions)))
        assert np.all(np.abs(contrib - want.contributions) <= tol), (
            f"expected {expected} against actual {actual}: contribution {worst} is "
            f"{contrib[worst]!r}, module 1 gives {want.contributions[worst]!r}; the breach report "
            "names a bin from these, so each one has to be right, not just their sum")
    got, _contrib = _psi_c([90, 5, 5], [98, 1, 1], 0.05)
    want = psi_from_counts_reference([90, 5, 5], [98, 1, 1], 0.05)
    assert abs(got - want.psi) <= psi_tolerance(want.contributions), (
        f"with floor 0.05, expected [90, 5, 5] against actual [98, 1, 1] gave psi {got!r}; "
        f"module 1's arithmetic gives {want.psi!r}. Floor at the `floor` argument you are "
        "given, not at the module's constant — section 8 calls this with other floors")
    empty, contrib = _psi_c([50, 50], [0, 0])
    assert math.isnan(empty) and np.all(np.isnan(contrib)), (
        f"with no actual records there are no shares: return NaN and set every contribution to "
        f"NaN; you returned {empty!r}")
    print("exercise 3 looks right: module 1's PSI, from counts, on every case")


if _IS_MAIN:
    _try("exercise 3", _check_psi_from_counts)

## 8. Exercise 4 — `breach_bin()` in `lesson.c`

A breach flag that cannot say where is a flag somebody has to investigate from scratch. Your
function decides whether the PSI breaches the threshold and names the bin with the largest
contribution — module 1's rule that a PSI exactly on the threshold is within it, a NaN that is
never a breach, and the lowest-numbered bin when two tie, so two monitors never disagree
about whom to blame.

<details><summary>💡 Hint 1 — what to think about</summary>

Two edge cases decide this one. What does "greater than the threshold" say about a PSI that
equals it, and about a NaN, which fails every ordered comparison? Write the test so that
BOTH of those land on "no breach" without a special case. Then the tie: scanning bins in
order and replacing the best only on a strictly larger contribution keeps the first of a tie.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

If it is not true that the PSI is strictly greater than the threshold, return minus one —
phrased that way round, a NaN falls into it on its own. Otherwise start with bin zero as the
best, walk the remaining bins in order, move the best only when a contribution is strictly
larger than the best so far, and return the best.

</details>

In [ ]:
def _breach_c(psi: float, contributions, threshold: float = PSI_THRESHOLD) -> int:
    return run_c("breach", "--psi", repr(float(psi)),
                 "--contrib", ",".join(repr(float(c)) for c in contributions),
                 "--threshold", repr(float(threshold)))["bin"]


def _check_breach_bin() -> None:
    cases = [
        (0.30, [0.05, 0.12, 0.12, 0.01], PSI_THRESHOLD, 1,
         "bins 1 and 2 tie for the largest contribution, and the LOWEST-numbered is named"),
        (0.25, [0.05, 0.12, 0.12, 0.01], PSI_THRESHOLD, -1,
         "a PSI exactly ON the threshold is within it — the test is strictly greater"),
        (math.nextafter(0.25, 1.0), [0.1, 0.0, 0.15], PSI_THRESHOLD, 2,
         "one representable step over the threshold is a breach"),
        (math.nan, [0.3, 0.2], PSI_THRESHOLD, -1,
         "a NaN PSI is greater than nothing, so it is not a breach — write the test so NaN "
         "fails it"),
        (0.2, [0.2, 0.0, 0.0], 0.1, 0,
         "bin 0 is a perfectly good answer: -1 means no breach, 0 means the first bin"),
        (0.9, [0.1, 0.4, 0.4], PSI_THRESHOLD, 1,
         "a tie at the end still goes to the lowest-numbered bin"),
        (0.1, [0.1, 0.0], PSI_THRESHOLD, -1, "under the threshold is no breach"),
    ]
    for psi, contributions, threshold, want, why in cases:
        got = _breach_c(psi, contributions, threshold)
        assert got == want, (f"breach_bin(psi={psi!r}, contributions={contributions}, "
                             f"threshold={threshold}) returned {got}, expected {want} — {why}")
    print("exercise 4 looks right")


if _IS_MAIN:
    _try("exercise 4", _check_breach_bin)

In [ ]:
def _show_the_floor() -> None:
    """Module 1's floor, from the monitor's side: what one emptied band does to the index."""
    expected, actual = [50, 30, 20], [70, 30, 0]   # a band that emptied out; nothing else moved
    floored, c_f = _psi_c(expected, actual, PSI_FLOOR)
    bare, c_b = _psi_c(expected, actual, 0.0)
    both_empty, _ = _psi_c([50, 50, 0], [100, 0, 0], 0.0)
    print(f"  development {expected} against this month {actual}")
    print(f"  floored at {PSI_FLOOR!r}:  psi {floored:.4f}, breach bin "
          f"{_breach_c(floored, c_f)} — the band that emptied is named")
    print(f"  unfloored:            psi {bare!r} — a number no report can rank or print")
    print(f"  and a bin empty on BOTH sides, unfloored: psi {both_empty!r}, breach bin "
          f"{_breach_c(both_empty, [0.0, 0.0, 0.0])}")
    print("  NaN is greater than nothing, so an unfloored monitor reports NO breach on exactly")
    print("  the month it most needed to. The floor is what keeps the finding.")


if _IS_MAIN:
    _try("the floor", _show_the_floor, needs=("exercise 3", "exercise 4"))

## 9. Exercise 5 — `reservoir_offer()` in `lesson.c`

The tail — the top two baseline deciles — is where the analyst goes to pull files, so the
monitor keeps a uniform random sample of `RESERVOIR_K` tail accounts. The tail's length is
unknown until the stream ends, and holding it all is exactly what the gate forbids, so the
sample is kept in O(k) memory by **Algorithm R** — Alan Waterman's, as Vitter's 1985 paper on
reservoir sampling credits it (`claims.yaml`).

The sample is evidence, so it must be **reproducible**: rerun the month and get the same
accounts. The random numbers therefore come from splitmix64, Vigna's four-line generator,
written once in C (given, in `lesson.c`) and once in Python (below) — so the notebook can
rerun your sample and compare it with yours slot by slot, bit for bit.

<details><summary>💡 Hint 1 — what to think about</summary>

Bit-for-bit means every detail of WHEN the generator is consulted matters, not just the
probabilities. The first k offers simply fill the slots, and consume no draws at all. After
that, each offer consumes exactly one draw, reduced modulo the number of offers seen
including this one — the classic off-by-one uses the count before this one — and replaces a
slot only when the result is strictly below k.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Read how many tail records were offered before this one. If that is still below k, write
this record into that slot. Otherwise draw once from the reservoir's generator, reduce it
modulo that count plus one as an unsigned 64-bit number, and if the result is below k,
overwrite that slot's account id and score. In both cases, finish by recording one more offer.

</details>

In [ ]:
def _reservoir_c(n: int, k: int, seed: int) -> dict:
    return run_c("reservoir", "--n", n, "--k", k, "--seed", seed)


def _check_reservoir_offer() -> None:
    _state, first = splitmix64_next(0)
    assert first == int(splitmix64_outputs(0, 1)[0]), "the two Python generators disagree"
    for n, k, seed in ((7, 20, 1), (20, 20, 5), (5000, 10, 20260917),
                       (100000, 1000, RESERVOIR_SEED)):
        rep = _reservoir_c(n, k, seed)
        ids = np.arange(1000, 1000 + n, dtype=np.uint64)
        scores = np.arange(n, dtype=np.float64) / n
        want_ids, want_scores, seen, state = reservoir_reference(ids, scores, k, seed)
        assert rep["seen"] == seen, (
            f"after {n} offers the reservoir says it has seen {rep['seen']}; count every offer "
            "once, whether or not it was kept")
        if rep["state"] != state:
            used = "every offer" if rep["state"] == (seed + n * GAMMA) & MASK64 else "some offers"
            raise AssertionError(
                f"n = {n}, k = {k}: the generator's state after the run is not what "
                f"{max(0, n - k)} draws leave. You drew on {used} — the first k offers fill "
                "slots 0..k-1 and must not draw at all, and every later offer draws exactly once")
        assert rep["outside_writes"] == 0, (
            f"n = {n}, k = {k}: {rep['outside_writes']} slot(s) beyond the first k were written. "
            "A draw that reduces to exactly k must not be stored anywhere — the test is "
            "strictly less than k, and slot k is not part of the sample")
        got_ids = np.array([sid for _j, sid, _s in rep["samples"]], dtype=np.uint64)
        got_scores = np.array([sc for _j, _sid, sc in rep["samples"]])
        assert got_ids.size == want_ids.size, (
            f"n = {n}, k = {k}: {got_ids.size} slots came back filled, expected {want_ids.size}")
        bad = np.nonzero(got_ids != want_ids)[0]
        assert bad.size == 0, (
            f"n = {n}, k = {k}, seed = {seed}: slot {bad[0]} holds account {got_ids[bad[0]]}, the "
            f"same algorithm in Python puts account {want_ids[bad[0]]} there ({bad.size} of "
            f"{k} slots differ). Check the modulus is the count INCLUDING this offer, and that "
            "a slot is replaced only when the draw reduces to strictly less than k")
        assert np.array_equal(got_scores, want_scores), (
            f"n = {n}, k = {k}: the right accounts, but a score is not the one offered with it")
    print("exercise 5 looks right: every sample identical to Python's, slot for slot")


if _IS_MAIN:
    print("splitmix64 seeded with 0, first three outputs, in Python:")
    _state = 0
    for _ in range(3):
        _state, _z = splitmix64_next(_state)
        print(f"  {_z:#018x}")
    _try("exercise 5", _check_reservoir_offer)

## 10. Exercise 6 — `monitor_stream()` in `lesson.c`

Now the job itself. Read the pipe through `g_buffer` — the only place a record may sit — and
for every complete 16-byte record: count it, reject it unless its score is a probability,
bin it, count it in its bin, offer it to the reservoir if it is in the tail, and call the
given `monitor_checkpoint()` every `CHECKPOINT_EVERY` records, which is what makes the PSI
a running one. Bytes that make no whole record at the end are a truncated record: report
them. The stub's comment has the full recipe.

The check below runs your monitor in **counts-only** mode — no tail, no PSI — so it judges
your loop on its own, before exercises 3 to 5 exist. It feeds the pipe in awkward 6001-byte
pieces with a pause between them, because a pipe hands a reader whatever has arrived so
far, and that is rarely a whole number of records. Then it streams the whole month through
your monitor and holds its peak memory to the ceiling from section 5 — with the tail sample
switched on as well, once exercise 5 passes.

<details><summary>💡 Hint 1 — what to think about</summary>

Four things are graded besides the counting. A NaN score must be rejected, and the natural
way to write "not between 0 and 1" lets NaN through, because every ordered comparison with
NaN is false. A read that returns fewer bytes than asked is the end of the stream, and
whatever does not make a whole record then is truncated. Reading whole records with a
record-sized element swallows that remainder silently. And nothing you keep may grow with
the stream.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Loop reading up to the buffer's size in bytes until a read returns nothing. Work out how many
bytes of each read make whole records, and walk them a record at a time with the given
decoder. Add one to the records read; if the score is at least zero AND at most one, bin it
and add one to that bin and to the valid count, and offer it to the reservoir when its bin is
at or above the profile's tail bin — otherwise add one to rejected. After each record, if the
records read is a multiple of the checkpoint interval, call the checkpoint. After each read,
add the leftover bytes to the truncated count. Return minus one if the stream reports an error.

</details>

In [ ]:
def _records(ids, scores) -> bytes:
    rec = np.empty(len(scores), dtype=REC_DTYPE)
    rec["account_id"] = np.asarray(ids, dtype=np.uint64)
    rec["score"] = np.asarray(scores, dtype=np.float64)
    return rec.tobytes()


def expected_counts(scores) -> dict:
    """What a correct monitor reports on `scores`, by module 1's rule, computed in numpy."""
    s = np.asarray(scores, dtype=np.float64)
    valid = (s >= 0.0) & (s <= 1.0)
    return {"records_read": s.size, "valid": int(valid.sum()), "rejected": int((~valid).sum()),
            "counts": np.bincount(module1_bins(s[valid], EDGES), minlength=N_BINS)}


def _monitor_counts(feed, write_size: int = 1 << 20, pause: float = 0.0) -> dict:
    return run_worker("monitor", "--profile", profile_file(), "--k", RESERVOIR_K, "--seed",
                      RESERVOIR_SEED, "--counts-only", feed=feed, write_size=write_size,
                      pause=pause)


def _compare_counts(rep: dict, want: dict, what: str) -> None:
    got_counts = np.array([rep["counts"][i] for i in range(N_BINS)])
    assert rep["records_read"] == want["records_read"], (
        f"{what}: records_read is {rep['records_read']}, the stream held {want['records_read']} "
        "whole records. If it is short, a read returned part of a record and the rest was "
        "lost: fread with an element size of 1 keeps reading until the buffer is full")
    assert rep["rejected"] == want["rejected"], (
        f"{what}: rejected is {rep['rejected']}, expected {want['rejected']}. If it is short by "
        "the number of NaN scores, your validity test lets NaN through — every comparison with "
        "NaN is false, so test that the score IS a probability rather than that it is not")
    assert rep["valid"] == want["valid"], (
        f"{what}: valid is {rep['valid']}, expected {want['valid']}")
    bad = np.nonzero(got_counts != want["counts"])[0]
    assert bad.size == 0, (
        f"{what}: bin {bad[0]} counted {got_counts[bad[0]]}, module 1's rule gives "
        f"{want['counts'][bad[0]]}. Count each valid record in the bin bin_index() returns")


def monitor_month(mode: str = "", limit=None) -> dict:
    """Stream the month (or its first `limit` bytes) through your monitor. Cached per build.

    `mode` is "" for the full monthly run, or one of the harness's switches: "--counts-only"
    (no tail, no PSI), "--no-psi" (the tail but no PSI) or "--no-tail" (the PSI but no tail).
    """
    build(verbose=False)
    key = ("month", mode, limit)
    if key not in _CACHE:
        _CACHE[key] = run_worker("monitor", "--profile", profile_file(), "--k", RESERVOIR_K,
                                 "--seed", RESERVOIR_SEED, *([mode] if mode else []),
                                 feed=month_file(), limit=limit)
    return _CACHE[key]


def _raw_units_per_mib() -> float:
    """How many raw ru_maxrss units one MiB is on this machine, MEASURED: a worker touches
    32 MiB and the rise is divided by 32. Cached per build. The check below uses it so that
    exercise 6 can be held to the memory ceiling before exercise 1 exists."""
    build(verbose=False)
    if "raw_per_mib" not in _CACHE:
        idle = run_worker("touch", "--mib", 0)["worker_maxrss_raw"]
        busy = run_worker("touch", "--mib", 32)["worker_maxrss_raw"]
        _CACHE["raw_per_mib"] = (busy - idle) / 32
    return _CACHE["raw_per_mib"]


def _check_memory_on_the_month() -> None:
    """Your monitor, counting the whole month — and, once exercise 5 passes, keeping its tail
    sample too — held to the ceiling: the worker's own baseline plus MEMORY_HEADROOM_MIB."""
    per_mib = _raw_units_per_mib()
    base = run_worker("baseline", "--profile", profile_file(), "--k", RESERVOIR_K, "--seed",
                      RESERVOIR_SEED)["worker_maxrss_raw"]
    modes = ["--counts-only"] + (["--no-psi"] if _STATUS.get("exercise 5") == "passed" else [])
    for mode in modes:
        rep = monitor_month(mode)
        _compare_counts(rep, REF, f"the month ({mode})")
        over = (rep["worker_maxrss_raw"] - base) / per_mib
        assert over <= MEMORY_HEADROOM_MIB, (
            f"streaming the month ({mode}), your monitor's worker peaked ≈{over:.1f} MiB above "
            f"its own baseline, against {MEMORY_HEADROOM_MIB:.0f} MiB of headroom. It keeps "
            "something that grows with the stream — an array of scores, a copy of the tail, a "
            "buffer that doubles. Keep the counts, the reservoir and g_buffer, and nothing else")


def _check_monitor_stream() -> None:
    head = MONTH[:5000].copy()
    specials = [math.nan, -1e-300, 1.0 + 2.0 ** -52, -0.0, 0.0, 1.0, math.inf, -math.inf,
                0.5, math.nan]
    head["score"][: len(specials)] = specials
    stray = b"\x07" * 11
    rep = _monitor_counts(_records(head["account_id"], head["score"]) + stray, write_size=6001,
                          pause=0.02)
    _compare_counts(rep, expected_counts(head["score"]), "5000 records with defects")
    assert rep["truncated_bytes"] == len(stray), (
        f"11 stray bytes followed the last whole record; truncated_bytes is "
        f"{rep['truncated_bytes']}. Read BYTES, and count what is left over after the last "
        "whole record — fread with a 16-byte element swallows a partial record silently")
    assert rep["checkpoints"] == 0, "5000 records is short of the first checkpoint"
    empty = _monitor_counts(b"")
    assert (empty["records_read"], empty["valid"], empty["truncated_bytes"]) == (0, 0, 0), (
        "an empty stream must leave every count at zero")
    longer = MONTH[:CHECKPOINT_EVERY + 5]
    rep = _monitor_counts(longer.tobytes())
    _compare_counts(rep, expected_counts(longer["score"]), f"{longer.size} records")
    assert rep["checkpoints"] == 1, (
        f"{longer.size} records passed one multiple of CHECKPOINT_EVERY, so monitor_checkpoint "
        f"must have been called exactly once; it was called {rep['checkpoints']} times")
    at, valid_then = rep["checkpoint_rows"][0][:2]
    want_valid = expected_counts(longer["score"][:CHECKPOINT_EVERY])["valid"]
    assert (at, valid_then) == (CHECKPOINT_EVERY, want_valid), (
        f"the checkpoint fired after record {at} with {valid_then} valid records counted; it "
        f"belongs right after record {CHECKPOINT_EVERY}, with {want_valid} counted — call it "
        "after the record has been counted, when records_read is a multiple of the interval")
    _check_memory_on_the_month()
    print("exercise 6 looks right: counts, rejects and truncation match, in one pass, and the "
          "whole month fits under the memory ceiling")


if _IS_MAIN:
    _try("exercise 6", _check_monitor_stream)

## 11. The month, streamed

Everything together: the whole month through the pipe, once, into your monitor, and every
figure it prints set against the in-memory reference from section 3 — the counts exactly,
the PSI within `psi_tolerance()`, the breach and the checkpoint that first saw it, and the
tail sample slot by slot.

Read the running PSI down the table. The monitor does not have to finish the month to know:
the checkpoint that first breaches is printed while the stream is still flowing, and it
names the bin. That is the difference between a monitor and a report.

In [ ]:
def _ulps_apart(a: float, b: float) -> int:
    return 0 if a == b else round(abs(a - b) / math.ulp(max(abs(a), abs(b))))


def _show_month_streamed() -> None:
    rep, ref = monitor_month(), REF
    _compare_counts(rep, ref, "the month")
    assert rep["truncated_bytes"] == 0, "the month is whole records; nothing is truncated"
    tol = psi_tolerance(ref["contributions"])
    assert abs(rep["psi"] - ref["psi"]) <= tol, (
        f"the month's PSI came back {rep['psi']!r} against module 1's {ref['psi']!r} "
        f"(tolerance {tol:.1e}), although every count agrees — so psi_from_counts is the part "
        "that differs")
    rows = rep["checkpoint_rows"]
    assert len(rows) == len(ref["checkpoints"]), (
        f"the monitor reported {len(rows)} checkpoints over {N_MONTH} records; there should be "
        f"{len(ref['checkpoints'])}, one every {CHECKPOINT_EVERY} records read")
    apart = [_ulps_apart(rep["psi"], ref["psi"])]
    print(f"  {'records read':>12}  {'valid':>8}  {'running PSI':>11}  {'module 1':>9}  breach")
    for (at, valid, psi, b), (r_at, r_valid, r_psi, r_b) in zip(rows, ref["checkpoints"]):
        assert (at, valid) == (r_at, r_valid), (
            f"a checkpoint fired at record {at} with {valid} valid; the reference has {r_at} "
            f"and {r_valid}")
        assert abs(psi - r_psi) <= PSI_TOLERANCE_ULPS * N_BINS * 2.0 ** -53 * max(1.0, r_psi), (
            f"the running PSI at record {at} is {psi!r}, module 1 gives {r_psi!r}")
        assert b == r_b, f"at record {at} the breach named bin {b}, the reference bin {r_b}"
        apart.append(_ulps_apart(psi, r_psi))
        flag = f"bin {b}" if b >= 0 else "-"
        if (at, b) == tuple(ref["first_breach"]):
            flag += "   <- first breach"
        print(f"  {at:>12}  {valid:>8}  {psi:>11.6f}  {r_psi:>9.6f}  {flag}")
    assert (rep["first_breach_at"], rep["first_breach_bin"]) == tuple(ref["first_breach"]), (
        f"first breach reported at {rep['first_breach_at']} (bin {rep['first_breach_bin']}); "
        f"the reference has {ref['first_breach']}")
    assert rep["breach_bin"] == ref["breach_bin"], (
        f"the month's breach names bin {rep['breach_bin']}, the reference bin {ref['breach_bin']}")
    b = rep["breach_bin"]
    print(f"\n  the month: PSI {rep['psi']:.6f} against module 1's {ref['psi']:.6f}; breach, "
          f"triggered by bin {b}")
    print(f"  bin {b} held {ref['expected_pct'][b]:.4f} of the development sample and "
          f"{ref['actual_pct'][b]:.4f} of this month")
    print(f"  every count identical; the {len(apart)} PSI figures within {max(apart)} unit(s) in "
          f"the last place of module 1's")
    got_ids = np.array([sid for _j, sid, _s in rep["samples"]], dtype=np.uint64)
    got_scores = np.array([sc for _j, _sid, sc in rep["samples"]])
    assert rep["tail_seen"] == ref["tail_seen"], (
        f"the reservoir was offered {rep['tail_seen']} tail records; there are "
        f"{ref['tail_seen']} valid records in bins {TAIL_BIN} and up")
    assert np.array_equal(got_ids, ref["sample_ids"]) and np.array_equal(
        got_scores, ref["sample_scores"]), (
        "the tail sample differs from the Python reference: offer every valid tail record, in "
        "stream order, and nothing else")
    print(f"  tail sample: {len(got_ids)} of {rep['tail_seen']} tail accounts, identical to "
          "Python's slot for slot; the first three:")
    for j in range(3):
        print(f"    slot {j}: account {got_ids[j]}, score {got_scores[j]:.6f}")


if _IS_MAIN:
    _try("the month, streamed", _show_month_streamed,
         needs=("exercise 2", "exercise 3", "exercise 4", "exercise 5", "exercise 6"))

## 12. The memory gate

Now the claim the whole lesson rests on, measured rather than promised. Your monitor, in
counts-only mode so that the gate can judge your loop before exercises 3 to 5 exist, streams
the first eighth, quarter, half and all of the month; beside it, the `hold` worker from
section 5 reads the same bytes. Once exercise 5 passes, the monitor is measured again with
its tail sample switched on, because a tail kept in a growing array is as much a copy of
the month as a buffer of scores. Every peak comes from `wait4()` on a forked worker,
converted by your `maxrss_mib()`, and set against the ceiling: the worker's own baseline
plus the headroom.

Read the monitor's column **down**: the input grows eightfold and the peak does not move.
The holder's column grows with the input, because it IS the input. Why the harness measures
a forked worker rather than the binary's own figure is the last line the cell prints.

In [ ]:
def _show_memory_gate() -> None:
    system = platform.system()
    base, ceiling = memory_ceiling()
    whole = monitor_month("--counts-only")
    _compare_counts(whole, REF, "the month, counts only")
    print(f"  baseline {base:.2f} MiB; ceiling {ceiling:.2f} MiB\n")
    print(f"  {'records':>9}  {'bytes streamed':>14}  {'your monitor':>12}  {'held in memory':>14}")
    peaks = []
    for parts in (8, 4, 2, 1):
        limit = N_MONTH // parts * RECORD_BYTES
        mon = whole if parts == 1 else monitor_month("--counts-only", limit=limit)
        held = run_worker("hold", feed=month_file(), limit=limit)
        peak = maxrss_mib(mon["worker_maxrss_raw"], system)
        peaks.append(peak)
        print(f"  {N_MONTH // parts:>9}  {limit:>14}  {peak:>8.2f} MiB  "
              f"{maxrss_mib(held['worker_maxrss_raw'], system):>10.2f} MiB")
    if _STATUS.get("exercise 5") == "passed":
        tail = monitor_month("--no-psi")
        _compare_counts(tail, REF, "the month, with the tail sample")
        peaks.append(maxrss_mib(tail["worker_maxrss_raw"], system))
        print(f"  {N_MONTH:>9}  {N_MONTH * RECORD_BYTES:>14}  {peaks[-1]:>8.2f} MiB  "
              "(your monitor again, now keeping its tail sample)")
    else:
        print("  your monitor with its tail sample is measured here too, once exercise 5 "
              "passes")
    worst = max(peaks)
    assert worst <= ceiling, (
        f"your monitor peaked at {worst:.2f} MiB against a ceiling of {ceiling:.2f} MiB. It "
        "keeps something that grows with the stream — an array of scores, a copy of the "
        "tail, a buffer that doubles. Keep counts, the reservoir and g_buffer, and nothing else")
    print(f"\n  PASS: {N_MONTH * RECORD_BYTES} bytes through a {STREAM_BUFFER_BYTES}-byte "
          f"buffer, at ≈{(worst - base) * 1024:.0f} KiB above the worker's own baseline")
    launcher = maxrss_mib(whole["launcher_maxrss_raw"], system)
    print(f"\n  the launching process's own high-water mark: {launcher:.2f} MiB; the forked "
          f"worker's, in the same run: {peaks[3]:.2f} MiB")
    print("  On Linux the kernel folds the address space an exec replaces into the new")
    print("  program's mark (claims.yaml), so the launcher's own figure starts at the size of")
    print("  the Python that started it. The worker is forked after exec and never held those")
    print("  pages. On macOS the two figures are close; on Linux they can be far apart.")


if _IS_MAIN:
    # Deliberately NOT waiting on exercise 6's check: when your monitor fails the ceiling,
    # this table is where you see why.
    _try("the memory gate", _show_memory_gate, needs=("exercise 1", "exercise 2"))

## 13. Common mistakes

**A binary built before you moved the checkout.** `make` decides a binary is up to date by
comparing its timestamp with its source's, and a binary that arrived with a copied or moved
directory is *newer* than the `lesson.c` beside it — so `make` will not rebuild it and you
will be grading yesterday's answers. A compiled lesson that links a library out of the
virtualenv is worse: the linker bakes in that library's absolute path, and after a move the
loader cannot find it at all. **Run `make clean` after moving or copying your checkout, or
after rebuilding the virtualenv.** This lesson links only the system C and maths libraries,
so only the stale-timestamp half can bite here — but the habit is the point, and the
repository's execution gate deletes compiled artefacts before grading any C lesson.

**`s < 0.0 || s > 1.0` as the test for "not a probability".** NaN fails every ordered
comparison, so it is not rejected — it goes into a bin. Test that the score *is* a
probability, and let everything else fall through to rejected.

**Feeding the monitor a file.** `./lesson_bin monitor < month.bin` hands it a regular file,
which can seek; the binary refuses, and says how to pipe it instead. The refusal is what
makes "one pass" a property of the harness rather than of your good intentions.

**`fread(g_buffer, RECORD_BYTES, n, in)`.** Reading whole records swallows a partial record
at the end of the stream without a word, and a truncated extract passes as complete.

**Keeping "just the tail".** The tail is a large share of this month (section 3 printed it).
A growing array of tail records is the month again, smaller; the reservoir exists so that
nothing does.

**Dividing `ru_maxrss` by 1024 on a Mac, or by 1024 twice on Linux**, and **trusting a
process's own `ru_maxrss` after exec** on Linux. Sections 4 and 12 measured both.

In [ ]:
def _demonstrate_common_mistakes() -> None:
    nan = math.nan
    print(f"    nan < 0.0 or nan > 1.0    -> {nan < 0.0 or nan > 1.0}   so the NaN is NOT "
          "rejected")
    print(f"    not (0.0 <= nan <= 1.0)   -> {not (0.0 <= nan <= 1.0)}    so the NaN IS rejected")
    with open(month_file(), "rb") as f:
        proc = subprocess.run([_binary(), "monitor", "--profile", str(profile_file())], stdin=f,
                              cwd=LESSON_DIR, capture_output=True, text=True, timeout=120)
    print(f"    the monitor handed the month as a FILE on stdin exits {proc.returncode}, and "
          "says:")
    print("      " + proc.stderr.strip().splitlines()[0][:96] + " ...")


if _IS_MAIN:
    _try("common mistakes", _demonstrate_common_mistakes)

## 14. Self-check

**1.** A colleague's monitor reproduces the reference's counts exactly and its PSI to the
last digit. It reads the month with a single `fread` into a buffer the size of the file.
Which part of this lesson's grading fails it?
(a) the PSI tolerance · (b) the counts comparison · (c) the peak-memory ceiling measured on
the worker · (d) none — a correct answer is a correct answer

**2.** The feed carries a NaN score. Which C test sends that record to `rejected`?
(a) `if (s < 0.0 || s > 1.0)` · (b) `if (!(s >= 0.0 && s <= 1.0))` ·
(c) `if (s < 0.0) ... else if (s > 1.0)` · (d) `if (isinf(s))`

**3.** The same worker's peak reads 34603008 on a Mac and 33792 on a Linux box. What do you
conclude?
(a) the Linux box used about a thousandth of the memory · (b) the Mac build leaks ·
(c) one of the two readings must be wrong · (d) both are 33 MiB: macOS counts bytes, Linux KiB

**4.** Why does the harness measure a forked worker rather than the launcher's own
`ru_maxrss`?
(a) on Linux a process's high-water mark survives exec, so the launcher's own figure starts
at the size of the Python that started it · (b) forking makes the stream faster ·
(c) `getrusage` cannot see static arrays · (d) a pipe needs two readers

In [ ]:
SELF_CHECK = {1: "?", 2: "?", 3: "?", 4: "?"}   # <- put a, b, c or d in each


# The marker holds a salted digest of each answer, not the answer. It still tells you at once
# which questions are wrong and which section settles each one, but reading this cell does not
# hand you the four letters. The reasoning is in this lesson's worked solution in the course repository.
_MARK = {
    1: ("249178727a10edf4dd2a69e4fff371bfb45935d21e83fe0259ca6274994363cf", "section 12"),
    2: ("e43f869d4ed49ce8ca7e48df579584efb01f9f7af02b6e841c48d5a032a3ca68", "section 10"),
    3: ("7bc1d1265355f35ee2ca4a39e95bb68ba4b0697163be116f375f8a99bce816ef", "section 4"),
    4: ("16f07d844440e4ed56ad66bdce79ee56a0e9185e28a0e7c8df26756736a2abd1", "section 12"),
}


def _check_self_check(answers: dict = None) -> None:
    """Mark the four multiple-choice answers, naming the section that settles each."""
    answers = SELF_CHECK if answers is None else answers
    wrong = [q for q, (want, _) in _MARK.items()
             if hashlib.sha256(
                 f"P04-L09:{q}:{str(answers.get(q, '?')).strip().lower()}".encode()
             ).hexdigest() != want]
    assert not wrong, (
        "questions " + ", ".join(str(q) for q in wrong) + " are still wrong. Look again at "
        + "; ".join(f"q{q}: {_MARK[q][1]}" for q in wrong) + ".")
    print("self-check: all four right")


def _self_check_marked() -> None:
    """Mark the quiz, or report it as not started while every answer is still '?'."""
    if all(str(v).strip() == "?" for v in SELF_CHECK.values()):
        raise NotImplementedError("put a, b, c or d against each question in SELF_CHECK, then "
                                  "re-run this cell")
    _check_self_check()


if _IS_MAIN:
    _try("self-check", _self_check_marked)

## 15. What you built

A monitor whose memory bound is **measured, not promised**: one pass through a pipe, a
fixed buffer, module 1's bins and floor, a running PSI that dates its own breach and names
the bin, and a tail sample anyone can regenerate from the seed — every figure matched against
an in-memory reference that was allowed the memory the monitor was not.

The breach, its bin, its checkpoint and its sample are exactly the findings module 10 turns
into the committee pack, each one traced back to the run that produced it.

In [ ]:
_MARKS = {"passed": "✅ passed", "failed": "❌ failed", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    print("YOUR PROGRESS")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        where = "lesson.c" if funcs[0] in _IN_C else "this notebook"
        print(f"  {_MARKS[state]:<15} {label:<10}  {funcs[0]}  ({where})")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    quiz = _MARKS[_STATUS.get("self-check", "not started")]
    print(f"\n  {done} of {len(_EXERCISES)} exercises complete · self-check: {quiz}")


if _IS_MAIN:
    # Re-run every exercise's check against lesson.c and your functions as they stand NOW, so
    # the board reports your latest edit, not whatever each cell said the last time you ran it.
    # Quietly: each check already printed its feedback in its own cell above.
    with contextlib.redirect_stdout(io.StringIO()):
        for _label, _check in (("exercise 1", _check_maxrss_mib),
                               ("exercise 2", _check_bin_index),
                               ("exercise 3", _check_psi_from_counts),
                               ("exercise 4", _check_breach_bin),
                               ("exercise 5", _check_reservoir_offer),
                               ("exercise 6", _check_monitor_stream),
                               ("self-check", _self_check_marked)):
            _try(_label, _check)
    _progress_board()
    if all(_STATUS.get(label) == "passed" for label in (*_EXERCISES, "self-check")):
        print("\n  all public checks green — now run:  python tools/grade.py <this lesson>")
    clean_up()
    print(f"\n  the temporary directory and the month's extract are deleted; wall time "
          f"{time.perf_counter() - _LESSON_T0:.1f} s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel it is a printed line, never a traceback.
    _still_failing = [label for label, state in _STATUS.items() if state == "failed"]
    if "ipykernel" in sys.modules:
        if _still_failing:
            print("\n  still failing: " + ", ".join(_still_failing)
                  + " — each one's message above names the likely mistake.")
    elif _FAILED_CHECKS:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))